# In-Silico Mutagenesis and Scrambled Controls

Learn how to use supremo_lite's mutagenesis functions to:
- Generate saturation mutagenesis sequences for every position in a region
- Create targeted mutations around specific anchors or BED-defined regions
- Generate scrambled control sequences that preserve nucleotide composition

## Setup

In [ ]:
import supremo_lite as sl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyfaidx import Fasta
from collections import Counter
import os

print(f"supremo_lite version: {sl.__version__}")

# Load test data
test_data_dir = "../../tests/data"
reference = Fasta(os.path.join(test_data_dir, "test_genome.fa"))

print(f"\nReference genome chromosomes:")
for chrom in reference.keys():
    print(f"  {chrom}: {len(reference[chrom])} bp")

## Saturation Mutagenesis with get_sm_sequences()

Generate all possible single-nucleotide mutations for every position in a region. Each position gets 3 alternate alleles (the nucleotides that differ from reference).

In [ ]:
# Mutate positions 10-20 on chr1
chrom = "chr1"
start, end = 10, 20

ref_seq, alt_seqs, metadata = sl.get_sm_sequences(
    chrom=chrom,
    start=start,
    end=end,
    reference_fasta=reference
)

print(f"Region: {chrom}:{start}-{end} ({end - start} bp)")
print(f"Reference sequence shape: {ref_seq.shape}")
print(f"Alternate sequences shape: {alt_seqs.shape}")
print(f"\nGenerated {len(metadata)} mutations (10 positions × 3 alternates)")
print(f"\nMetadata columns: {list(metadata.columns)}")
print(f"\nFirst 6 mutations:")
print(metadata.head(6).to_string(index=False))

## Targeted Mutagenesis with get_sm_subsequences()

Mutate only specific regions within a larger sequence window. Two approaches:
1. **Anchor-based**: Center mutations around a specific position
2. **BED-based**: Mutate regions defined in a BED file

In [ ]:
# Anchor-based: mutate positions within ±5 bp of position 40
ref_seq, alt_seqs, metadata = sl.get_sm_subsequences(
    chrom="chr1",
    seq_len=80,           # Full window length
    reference_fasta=reference,
    anchor=40,            # Center position
    anchor_radius=5       # Mutate positions 35-45
)

print(f"Anchor-based mutagenesis:")
print(f"  Window: 80 bp centered on position 40")
print(f"  Mutation region: positions 35-45 (±5 bp of anchor)")
print(f"  Generated {len(metadata)} mutations (10 positions × 3 alternates)")
print(f"\nMutated positions (variant_offset0):")
print(f"  {sorted(metadata['variant_offset0'].unique())}")

In [ ]:
# BED-based: mutate regions from a BED file
bed_df = pd.DataFrame({
    "chrom": ["chr1", "chr1"],
    "start": [10, 50],
    "end": [15, 55]
})

ref_seq, alt_seqs, metadata = sl.get_sm_subsequences(
    chrom="chr1",
    seq_len=80,
    reference_fasta=reference,
    bed_regions=bed_df
)

print(f"BED-based mutagenesis:")
print(f"  Region 1: chr1:10-15 (5 bp)")
print(f"  Region 2: chr1:50-55 (5 bp)")
print(f"  Generated {len(metadata)} mutations (10 positions × 3 alternates)")

## Scrambled Control Sequences

The `get_scrambled_subsequences()` function generates negative control sequences by scrambling BED-defined regions while **preserving nucleotide composition**. This is useful for:
- Creating matched controls that maintain GC content
- Disrupting regulatory motifs while keeping sequence properties
- Generating multiple scrambled versions for statistical analysis

In [ ]:
# Define a region to scramble
bed_df = pd.DataFrame({
    "chrom": ["chr1"],
    "start": [20],
    "end": [60]      # 40 bp region to scramble
})

# Generate 5 scrambled versions
ref_seqs, scrambled_seqs, metadata = sl.get_scrambled_subsequences(
    chrom="chr1",
    seq_len=80,
    reference_fasta=reference,
    bed_regions=bed_df,
    n_scrambles=5,
    random_state=42     # For reproducibility
)

print(f"Scrambled subsequences:")
print(f"  Reference sequences: {ref_seqs.shape}")
print(f"  Scrambled sequences: {scrambled_seqs.shape}")
print(f"  Metadata rows: {len(metadata)}")
print(f"\nMetadata columns: {list(metadata.columns)}")

In [ ]:
# Examine the metadata
print("Scramble metadata:")
print(metadata.to_string(index=False))

## Nucleotide Composition Preservation

The key property of scrambling is that it preserves nucleotide composition while disrupting sequence patterns.

In [ ]:
# Get original and scrambled sequences from metadata
original = metadata.iloc[0]["original_seq"]
scrambled = metadata.iloc[0]["scrambled_seq"]

print(f"Original sequence:  {original}")
print(f"Scrambled sequence: {scrambled}")
print(f"\nLength preserved: {len(original)} == {len(scrambled)}")

# Count nucleotides
orig_counts = Counter(original)
scram_counts = Counter(scrambled)

print(f"\nNucleotide counts:")
print(f"  Original:  A={orig_counts['A']}, C={orig_counts['C']}, G={orig_counts['G']}, T={orig_counts['T']}")
print(f"  Scrambled: A={scram_counts['A']}, C={scram_counts['C']}, G={scram_counts['G']}, T={scram_counts['T']}")
print(f"\nComposition preserved: {orig_counts == scram_counts}")

## Visualizing Encoded Sequences

Compare the one-hot encoded reference and scrambled sequences to see how the scrambling affects the sequence pattern.

In [ ]:
# Get the scramble region boundaries
scram_start = int(metadata.iloc[0]['scramble_start'])
scram_end = int(metadata.iloc[0]['scramble_end'])

# Convert to numpy for visualization
ref_viz = ref_seqs[0].numpy() if hasattr(ref_seqs[0], 'numpy') else ref_seqs[0]
scram_viz = scrambled_seqs[0].numpy() if hasattr(scrambled_seqs[0], 'numpy') else scrambled_seqs[0]

fig, axes = plt.subplots(2, 1, figsize=(14, 5))

# Reference sequence
im1 = axes[0].imshow(ref_viz, cmap='Blues', aspect='auto')
axes[0].set_yticks([0, 1, 2, 3])
axes[0].set_yticklabels(['A', 'C', 'G', 'T'])
axes[0].set_title('Reference Sequence')
axes[0].axvline(x=scram_start, color='red', linestyle='--', linewidth=2, label='Scramble region')
axes[0].axvline(x=scram_end, color='red', linestyle='--', linewidth=2)
axes[0].axvspan(scram_start, scram_end, alpha=0.1, color='red')
axes[0].legend(loc='upper right')

# Scrambled sequence
im2 = axes[1].imshow(scram_viz, cmap='Oranges', aspect='auto')
axes[1].set_yticks([0, 1, 2, 3])
axes[1].set_yticklabels(['A', 'C', 'G', 'T'])
axes[1].set_title('Scrambled Sequence (same region highlighted)')
axes[1].set_xlabel('Position')
axes[1].axvline(x=scram_start, color='red', linestyle='--', linewidth=2)
axes[1].axvline(x=scram_end, color='red', linestyle='--', linewidth=2)
axes[1].axvspan(scram_start, scram_end, alpha=0.1, color='red')

plt.tight_layout()
plt.show()

print(f"Scrambled region: positions {scram_start}-{scram_end}")
print(f"Regions outside the scramble boundaries remain identical.")

## Reproducibility with random_state

Use `random_state` to ensure scrambled sequences are reproducible across runs.

In [ ]:
# Run twice with same random_state
_, _, meta1 = sl.get_scrambled_subsequences(
    chrom="chr1", seq_len=80, reference_fasta=reference,
    bed_regions=bed_df, n_scrambles=3, random_state=123
)

_, _, meta2 = sl.get_scrambled_subsequences(
    chrom="chr1", seq_len=80, reference_fasta=reference,
    bed_regions=bed_df, n_scrambles=3, random_state=123
)

print("Same random_state produces identical scrambles:")
for i in range(3):
    match = meta1.iloc[i]['scrambled_seq'] == meta2.iloc[i]['scrambled_seq']
    print(f"  Scramble {i}: {meta1.iloc[i]['scrambled_seq'][:20]}... == {meta2.iloc[i]['scrambled_seq'][:20]}... : {match}")

## Next Steps

- **[01_getting_started.ipynb](01_getting_started.ipynb)** - Basic supremo_lite functionality
- **[02_personalized_genomes.ipynb](02_personalized_genomes.ipynb)** - Genome personalization workflows
- **[03_prediction_alignment.ipynb](03_prediction_alignment.ipynb)** - Align model predictions across variants
- **[04_pam_disruption.ipynb](04_pam_disruption.ipynb)** - CRISPR PAM disruption analysis
- **[Mutagenesis Guide](../user_guide/mutagenesis.md)** - Detailed documentation and API reference